# Embedding và Vector Storage — E5 + Qdrant

Hiện thực một luồng ingestion đơn giản:

**Output Chunker → kiểm tra đầu vào → E5 embedding → Qdrant → kiểm tra dữ liệu đã lưu.**

E5 tạo vector; Qdrant lưu vector cùng nội dung và metadata nguồn của chunk.
Chạy từ trên xuống với kernel `.venv`. Dependency nằm trong `requirements.txt`;
model có thể được tải từ Hugging Face ở lần chạy đầu.

Tài liệu: [E5](https://huggingface.co/intfloat/multilingual-e5-small),
[Qdrant points](https://qdrant.tech/documentation/manage-data/points/).

## Setup

Import thư viện, sau đó đặt đường dẫn input, database và model dùng để embedding.
Database được lưu trong `output/qdrant_db`.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

In [2]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

INPUT_PATH = ROOT / "output/chunks/structure_aware_recursive.jsonl"
DB_PATH = ROOT / "output/qdrant_db"
MODEL_NAME = "intfloat/multilingual-e5-small"
MODEL_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
torch.set_num_threads(4)

## Đọc output Chunker

Đọc các dòng JSONL, rồi dựng lại `Document(page_content, metadata)`.
Giữ nguyên text và metadata từ Chunker để sau này truy vết nguồn.

In [3]:
with INPUT_PATH.open(encoding="utf-8") as stream:
    rows = [json.loads(line) for line in stream if line.strip()]

In [4]:
chunks = [
    Document(page_content=row["page_content"], metadata=row["metadata"])
    for row in rows
]
len(chunks)

241

### Chọn chunk có nguồn và xem một mẫu

Bỏ các chunk chỉ chứa heading, không có `sources`. `indices` giữ vị trí dòng gốc;
chúng cũng sẽ là ID của point trong collection này.
Cell sau hiển thị một chunk để biết nội dung đầu vào đang được xử lý.

In [5]:
indices = [
    index for index, chunk in enumerate(chunks)
    if chunk.metadata["sources"]
]
assert indices, "Không có chunk có nguồn để embedding."
len(indices)

233

In [6]:
pd.Series({
    "row_index": indices[0],
    "chapter": chunks[indices[0]].metadata["chapter_number"],
    "section": chunks[indices[0]].metadata["section_number"],
    "text_preview": chunks[indices[0]].page_content[:240],
})

row_index                                                       0
chapter                                                         1
section                                                   1.intro
text_preview    [[SECTION 1.intro: Chapter Introduction]]\nsto...
dtype: object

## Tạo embedding bằng E5

Nạp model trên CPU. E5-small tạo một vector 384 chiều cho mỗi chunk;
revision cố định giúp dùng đúng cùng phiên bản model khi chạy lại.

In [7]:
model = SentenceTransformer(
    MODEL_NAME, revision=MODEL_REVISION, device="cpu",
)
model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

### Chuẩn bị text đầu vào

E5 cần tiền tố `passage: ` cho tài liệu. Chỉ thêm vào `inputs` dùng để encode;
`page_content` và các offset trong metadata vẫn giữ nguyên.

In [8]:
texts = [chunks[index].page_content for index in indices]
assert all(text.strip() for text in texts), "Có chunk có nội dung rỗng."
inputs = ["passage: " + text for text in texts]
inputs[0][:160]

'passage: [[SECTION 1.intro: Chapter Introduction]]\nstore images, audio clips, and video streams digitally. These types of files are becoming an important compon'

### Kiểm tra giới hạn token

Đếm bằng tokenizer của chính model, sau khi thêm tiền tố. Không cắt text khi đếm;
nếu có đầu vào vượt giới hạn thì dừng trước encode để tránh mất nội dung.

In [9]:
token_counts = [
    len(model.tokenizer(text, truncation=False)["input_ids"])
    for text in inputs
]
assert max(token_counts) <= model.max_seq_length, "Có chunk vượt giới hạn token."
pd.Series(token_counts, name="input_tokens").describe()

count    233.000000
mean     161.210300
std       54.814605
min       27.000000
25%      121.000000
50%      170.000000
75%      203.000000
max      296.000000
Name: input_tokens, dtype: float64

### Encode và kiểm tra vector

Encode các chunk theo batch nhỏ và chuẩn hóa L2. Dòng `.shape` cho biết số vector
và số chiều. Cell tiếp theo kiểm tra vector hữu hạn, đủ số lượng và có độ dài gần 1.

In [10]:
vectors = model.encode(
    inputs, batch_size=8, precision="float32", normalize_embeddings=True,
    prompt="", show_progress_bar=False,
)
vectors.shape

(233, 384)

In [11]:
assert vectors.shape == (len(indices), 384)
assert np.isfinite(vectors).all()
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-5)

## Chuẩn bị Qdrant

Tên collection được tạo từ model, revision và nội dung input. Cùng đầu vào tạo cùng tên;
đổi input/model thì tạo collection khác, tránh trộn dữ liệu của các lần chạy.
Sau đó mở Qdrant local trên đĩa; không cần chạy server.

In [12]:
key = MODEL_NAME + MODEL_REVISION + INPUT_PATH.read_text(encoding="utf-8")
COLLECTION_NAME = "e5_" + hashlib.sha256(key.encode()).hexdigest()[:12]
COLLECTION_NAME

'e5_183a932fed56'

In [13]:
if "client" in globals():
    client.close()
client = QdrantClient(path=str(DB_PATH))

### Tạo collection và kiểm tra cấu hình

Collection cần biết số chiều vector và loại khoảng cách. Dùng số chiều từ output E5
và `COSINE`. Nếu collection đã có thì dùng lại; cell sau kiểm tra cấu hình khớp.

In [14]:
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=vectors.shape[1], distance=models.Distance.COSINE,
        ),
    )

In [15]:
config = client.get_collection(COLLECTION_NAME).config.params.vectors
assert config.size == vectors.shape[1]
assert config.distance == models.Distance.COSINE

## Ghép chunk thành point

Một point gồm **ID + vector + payload**. ID là vị trí dòng gốc; vector chuyển sang
list để gửi vào Qdrant. Payload lưu `page_content` và toàn bộ `metadata`, gồm `sources`.

In [16]:
points = [
    models.PointStruct(
        id=indices[position], vector=vector.tolist(),
        payload={"page_content": chunks[indices[position]].page_content,
                 "metadata": chunks[indices[position]].metadata},
    )
    for position, vector in enumerate(vectors)
]
len(points)

233

### Lưu bằng upsert và đếm point

`upsert` thêm point mới hoặc cập nhật point có cùng ID. Chạy lại cùng dữ liệu sẽ dùng
lại collection và các ID. `wait=True` chờ thao tác ghi hoàn tất trước khi kiểm tra số point.

In [17]:
client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [18]:
stored_count = client.count(collection_name=COLLECTION_NAME, exact=True).count
assert stored_count == len(points)
stored_count

233

## Kiểm tra dữ liệu sau khi mở lại

Lần lượt: đóng client → mở lại cùng database → đọc các point theo ID → đối chiếu dữ liệu.
Đọc theo ID giúp kiểm tra vector, text và metadata đã được lưu trên đĩa.
Cuối cùng xem một point mẫu và đóng client để giải phóng database.

In [19]:
client.close()

In [20]:
client = QdrantClient(path=str(DB_PATH))

In [21]:
saved = client.retrieve(
    collection_name=COLLECTION_NAME, ids=[point.id for point in points],
    with_payload=True, with_vectors=True,
)

In [22]:
expected = {point.id: point for point in points}
assert len(saved) == len(points)
for point in saved:
    assert point.payload == expected[point.id].payload
    assert np.allclose(point.vector, expected[point.id].vector, atol=1e-6)
pd.Series({"points_checked": len(saved), "vectors_match": True, "payload_match": True})

points_checked     233
vectors_match     True
payload_match     True
dtype: object

In [23]:
pd.Series({
    "point_id": saved[0].id,
    "vector_dimensions": len(saved[0].vector),
    "source_count": len(saved[0].payload["metadata"]["sources"]),
    "text_preview": saved[0].payload["page_content"][:240],
})

point_id                                                             0
vector_dimensions                                                  384
source_count                                                         1
text_preview         [[SECTION 1.intro: Chapter Introduction]]\nsto...
dtype: object

In [24]:
client.close()